# Task 1: Clean nav_history.csv

In [1]:

import pandas as pd
from pathlib import Path

# Setup dynamic paths
project_dir = Path.cwd().parent
raw_data_path = project_dir / 'data' / 'raw' / '02_nav_history.csv'
processed_dir = project_dir / 'data' / 'processed'
output_path = processed_dir / 'clean_nav.csv'

processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_data_path)

# 1. Parse dates to datetime
df['date'] = pd.to_datetime(df['date'])

# 2. Remove duplicates
df = df.drop_duplicates(subset=['amfi_code', 'date'])

original_count = len(df)

# 3. Create a complete date range and forward-fill missing NAVs
# Set index to date to use resample
df = df.set_index('date')

# Group by amfi_code, resample to daily ('D'), and forward-fill
df_filled = df.groupby('amfi_code')['nav'].resample('D').ffill().reset_index()

# Sort by amfi_code and date
df_filled = df_filled.sort_values(by=['amfi_code', 'date'])

filled_count = len(df_filled)
added_rows = filled_count - original_count

print(f"Original rows: {original_count}")
print(f"Rows after filling holidays/weekends: {filled_count}")
print(f"Added {added_rows} new rows with forward-filled NAVs for weekends and holidays.")

# 4. Validate NAV > 0
df_filled = df_filled[df_filled['nav'] > 0]

# Save to CSV
df_filled.to_csv(output_path, index=False)
print(f"Cleaned data saved to {output_path}")


Original rows: 46000
Rows after filling holidays/weekends: 64320
Added 18320 new rows with forward-filled NAVs for weekends and holidays.
Cleaned data saved to C:\programming\bluestock_mf_capstone\data\processed\clean_nav.csv


# Task 2: Clean investor_transactions.csv

In [2]:

import pandas as pd
from pathlib import Path

# Setup paths
project_dir = Path.cwd().parent
raw_data_path = project_dir / 'data' / 'raw' / '08_investor_transactions.csv'
processed_dir = project_dir / 'data' / 'processed'
output_path = processed_dir / 'clean_transactions.csv'

df = pd.read_csv(raw_data_path)

# Show original counts
print(f"Original transaction rows: {len(df)}")

# 1. Fix date formats
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# 2. Standardise transaction_type
df['transaction_type'] = df['transaction_type'].str.strip().str.title()
# Explicitly ensure SIP is uppercase as it's an acronym
df['transaction_type'] = df['transaction_type'].replace({'Sip': 'SIP'}) 
print(f"Standardised transaction types: {df['transaction_type'].unique()}")

# 3. Validate amount_inr > 0
invalid_amounts = len(df[df['amount_inr'] <= 0])
if invalid_amounts > 0:
    print(f"Removing {invalid_amounts} rows with amount <= 0")
df = df[df['amount_inr'] > 0]

# 4. Check and standardise KYC status values
df['kyc_status'] = df['kyc_status'].str.strip().str.title()
print(f"Standardised KYC statuses found: {df['kyc_status'].unique()}")

# Save
df.to_csv(output_path, index=False)
print(f"Cleaned transaction data saved to {output_path}")
print(f"Final transaction rows: {len(df)}")


Original transaction rows: 32778
Standardised transaction types: <StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
Standardised KYC statuses found: <StringArray>
['Verified', 'Pending']
Length: 2, dtype: str


Cleaned transaction data saved to C:\programming\bluestock_mf_capstone\data\processed\clean_transactions.csv
Final transaction rows: 32778


# Task 3: Clean scheme_performance.csv

In [3]:

import pandas as pd
from pathlib import Path

# Setup paths
project_dir = Path.cwd().parent
raw_data_path = project_dir / 'data' / 'raw' / '07_scheme_performance.csv'
processed_dir = project_dir / 'data' / 'processed'
output_path = processed_dir / 'clean_performance.csv'

df = pd.read_csv(raw_data_path)

print(f"Original performance rows: {len(df)}")

# 1. Validate return values are numeric
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows if any return value became NaN due to coercion (meaning it wasn't numeric)
missing_returns = df[return_cols].isna().any(axis=1).sum()
if missing_returns > 0:
    print(f"Removing {missing_returns} rows with invalid/non-numeric return values")
    df = df.dropna(subset=return_cols)

# 2. Flag negative Sharpe ratios
df['sharpe_ratio'] = pd.to_numeric(df['sharpe_ratio'], errors='coerce')
df['has_negative_sharpe'] = df['sharpe_ratio'] < 0
negative_sharpe_count = df['has_negative_sharpe'].sum()
print(f"Flagged {negative_sharpe_count} funds with a negative Sharpe ratio")

# 3. Check expense_ratio range (0.1% - 2.5%) and filter
df['expense_ratio_pct'] = pd.to_numeric(df['expense_ratio_pct'], errors='coerce')
valid_expense = (df['expense_ratio_pct'] >= 0.1) & (df['expense_ratio_pct'] <= 2.5)
invalid_expense_count = (~valid_expense).sum()
print(f"Removing {invalid_expense_count} rows outside the expense ratio range of 0.1% - 2.5%")
df = df[valid_expense]

# Save
df.to_csv(output_path, index=False)
print(f"Cleaned performance data saved to {output_path}")
print(f"Final performance rows: {len(df)}")


Original performance rows: 40
Flagged 0 funds with a negative Sharpe ratio
Removing 0 rows outside the expense ratio range of 0.1% - 2.5%
Cleaned performance data saved to C:\programming\bluestock_mf_capstone\data\processed\clean_performance.csv
Final performance rows: 40


# Task 3.5: Clean Remaining Datasets

In [4]:

import pandas as pd
from pathlib import Path

project_dir = Path.cwd().parent
raw_dir = project_dir / 'data' / 'raw'
processed_dir = project_dir / 'data' / 'processed'

remaining_files = [
    '01_fund_master.csv',
    '03_aum_by_fund_house.csv',
    '04_monthly_sip_inflows.csv',
    '05_category_inflows.csv',
    '06_industry_folio_count.csv',
    '09_portfolio_holdings.csv',
    '10_benchmark_indices.csv'
]

for file in remaining_files:
    raw_path = raw_dir / file
    out_path = processed_dir / f"clean_{file.split('_', 1)[1]}"
    
    if not raw_path.exists():
        continue
        
    print(f"\n--- Processing {file} ---")
    df = pd.read_csv(raw_path)
    original_len = len(df)
    
    # 1. Duplicates
    df = df.drop_duplicates()
    
    # 2. Date Parsing
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'month' in c.lower()]
    for c in date_cols:
        if df[c].dtype == 'object':
            df[c] = pd.to_datetime(df[c], errors='coerce')
            nat_count = df[c].isna().sum()
            if nat_count > 0:
                df = df.dropna(subset=[c])
    
    # 3. Missing Values - We now KEEP them instead of dropping!
    null_counts = df.isna().any(axis=1).sum()
    if null_counts > 0:
        print(f"Found {null_counts} rows with missing (null) values. Keeping them as Nulls to preserve data integrity (e.g. missing YoY data).")
    else:
        print("No missing (null) values found.")
        
    # Save
    df.to_csv(out_path, index=False)
    print(f"Saved to {out_path.name}. Final rows: {len(df)}")



--- Processing 01_fund_master.csv ---
No missing (null) values found.
Saved to clean_fund_master.csv. Final rows: 40

--- Processing 03_aum_by_fund_house.csv ---
No missing (null) values found.
Saved to clean_aum_by_fund_house.csv. Final rows: 90

--- Processing 04_monthly_sip_inflows.csv ---
Found 12 rows with missing (null) values. Keeping them as Nulls to preserve data integrity (e.g. missing YoY data).
Saved to clean_monthly_sip_inflows.csv. Final rows: 48

--- Processing 05_category_inflows.csv ---
No missing (null) values found.
Saved to clean_category_inflows.csv. Final rows: 144

--- Processing 06_industry_folio_count.csv ---
No missing (null) values found.
Saved to clean_industry_folio_count.csv. Final rows: 21

--- Processing 09_portfolio_holdings.csv ---
No missing (null) values found.
Saved to clean_portfolio_holdings.csv. Final rows: 322

--- Processing 10_benchmark_indices.csv ---
No missing (null) values found.


Saved to clean_benchmark_indices.csv. Final rows: 8050
